# Clean EDGAR data for the globe

Build three tables for the country globe, the emissions time series, and the sector breakdown.

- `data/cleaned/country_totals.csv` — one row per country and year, all greenhouse gases, Mt CO2eq
- `data/cleaned/country_sectors.csv` — one row per country, year, sector, and gas, Mt CO2eq
- `data/cleaned/country_snapshot.csv` — one row per country for the latest year, with sector amounts and shares

International aviation, international shipping, EU27, and the global total are dropped because they are not countries on the globe.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "raw" / "GHG_totals_by_country.csv").exists():
    ROOT = ROOT.parent

RAW = ROOT / "raw"
OUT = ROOT / "data" / "cleaned"
OUT.mkdir(parents=True, exist_ok=True)

GAS_NAMES = {
    "CO2": "CO2",
    "GWP_100_AR5_CH4": "CH4",
    "GWP_100_AR5_N2O": "N2O",
    "GWP_100_AR5_F-gases": "F-gases",
}

NON_COUNTRY_CODES = {"AIR", "SEA", "EU27", "GLOBAL TOTAL"}
ID_COLS = ["EDGAR Country Code", "Country"]


def year_columns(frame: pd.DataFrame) -> list[str]:
    return [column for column in frame.columns if str(column).isdigit()]


def load_countries(name: str) -> pd.DataFrame:
    frame = pd.read_csv(RAW / f"{name}.csv")
    frame = frame.dropna(subset=["Country", "EDGAR Country Code"])
    return frame[~frame["EDGAR Country Code"].isin(NON_COUNTRY_CODES)].copy()


def to_long(frame: pd.DataFrame, value_name: str, extra_id: list[str]) -> pd.DataFrame:
    long = frame.melt(
        id_vars=ID_COLS + extra_id,
        value_vars=year_columns(frame),
        var_name="year",
        value_name=value_name,
    )
    long["year"] = long["year"].astype(int)
    long[value_name] = pd.to_numeric(long[value_name], errors="coerce").round(6)
    long = long.rename(columns={"EDGAR Country Code": "country_code", "Country": "country"})
    return long.dropna(subset=[value_name])


## Country totals

The totals sheet is already all gases combined, in Mt CO2eq per year. Reshape year columns into rows so a year filter can read one series per country.

In [2]:
totals = to_long(load_countries("GHG_totals_by_country"), "mt_co2eq", [])
totals.insert(0, "unit", "Mt CO2eq/yr")
totals.to_csv(OUT / "country_totals.csv", index=False)

totals.head()


,unit,country_code,country,year,mt_co2eq
0,Mt CO2eq/yr,ABW,Aruba,1970,0.038315
1,Mt CO2eq/yr,AFG,Afghanistan,1970,15.445148
2,Mt CO2eq/yr,AGO,Angola,1970,18.925319
3,Mt CO2eq/yr,AIA,Anguilla,1970,0.003563
4,Mt CO2eq/yr,ALB,Albania,1970,8.288397


## Sectors and gases

Keep each sector and gas so the side panel can filter. Gas names are shortened to CO2, CH4, N2O, and F-gases. Values stay in Mt CO2eq per year, using the 100-year global warming potentials already applied in the source file.

In [3]:
sectors = load_countries("GHG_by_sector_and_country")
sectors = sectors.dropna(subset=["Substance", "Sector"])
sectors["Substance"] = sectors["Substance"].map(GAS_NAMES)
if sectors["Substance"].isna().any():
    missing = sectors.loc[sectors["Substance"].isna()].index.tolist()[:5]
    raise ValueError(f"Unmapped gas names at rows {missing}")

sectors = to_long(sectors, "mt_co2eq", ["Substance", "Sector"])
sectors = sectors.rename(columns={"Substance": "gas", "Sector": "sector"})
sectors.insert(0, "unit", "Mt CO2eq/yr")
sectors.to_csv(OUT / "country_sectors.csv", index=False)

sectors.head()


,unit,country_code,country,gas,sector,year,mt_co2eq
0,Mt CO2eq/yr,AFG,Afghanistan,CO2,Agriculture,1970,0.029229
1,Mt CO2eq/yr,ALB,Albania,CO2,Agriculture,1970,0.113300
2,Mt CO2eq/yr,ARG,Argentina,CO2,Agriculture,1970,0.109371
3,Mt CO2eq/yr,ARM,Armenia,CO2,Agriculture,1970,0.055288
4,Mt CO2eq/yr,AUS,Australia,CO2,Agriculture,1970,0.311143


## Latest-year snapshot

One row per country for the side panel: the total, each sector summed across gases, and that sector's share of the country total. Shares are for the bar chart.

In [4]:
latest = int(totals["year"].max())
snap_totals = totals.loc[totals["year"] == latest, ["country_code", "country", "mt_co2eq"]]
snap_sectors = (
    sectors.loc[sectors["year"] == latest]
    .groupby(["country_code", "sector"], as_index=False)["mt_co2eq"]
    .sum()
)
wide = snap_sectors.pivot(index="country_code", columns="sector", values="mt_co2eq")
wide = wide.rename(columns=lambda name: "sector_" + name.lower().replace(" ", "_"))

snapshot = snap_totals.merge(wide, on="country_code", how="left")
sector_cols = [column for column in snapshot.columns if column.startswith("sector_")]
snapshot["year"] = latest
snapshot["unit"] = "Mt CO2eq/yr"
for column in sector_cols:
    snapshot[column.replace("sector_", "share_")] = (snapshot[column] / snapshot["mt_co2eq"]).round(6)
    snapshot[column] = snapshot[column].round(6)

front = ["country_code", "country", "year", "unit", "mt_co2eq"]
share_cols = [column.replace("sector_", "share_") for column in sector_cols]
snapshot = snapshot[front + sector_cols + share_cols].sort_values("mt_co2eq", ascending=False)
snapshot.to_csv(OUT / "country_snapshot.csv", index=False)

print(f"{snapshot['country_code'].nunique()} countries, latest year {latest}")
print(f"totals {len(totals):,} rows, sectors {len(sectors):,} rows")
snapshot.head()


208 countries, latest year 2025
totals 11,648 rows, sectors 263,243 rows


,country_code,country,year,unit,mt_co2eq,sector_agriculture,sector_buildings,sector_fuel_production,sector_industrial_combustion,sector_power_industry,...,sector_transport,sector_waste,share_agriculture,share_buildings,share_fuel_production,share_industrial_combustion,share_power_industry,share_processes,share_transport,share_waste
35,CHN,China,2025,Mt CO2eq/yr,15980.974041,762.989956,611.077067,1517.115182,2675.285188,6634.098029,...,1119.757535,302.794588,0.047744,0.038238,0.094933,0.167404,0.415125,0.147541,0.070068,0.018947
196,USA,United States,2025,Mt CO2eq/yr,6018.301241,397.938082,550.223809,717.647804,457.613364,1573.493615,...,1750.031333,174.327840,0.066121,0.091425,0.119244,0.076037,0.261451,0.065970,0.290785,0.028966
89,IND,India,2025,Mt CO2eq/yr,4508.007756,834.174063,250.295022,218.712494,681.770927,1571.652840,...,381.696852,158.678649,0.185043,0.055522,0.048516,0.151236,0.348636,0.091177,0.084671,0.035199
159,RUS,Russia,2025,Mt CO2eq/yr,2574.389406,118.152829,225.971354,490.305111,278.381829,892.216624,...,284.115038,101.766008,0.045895,0.087777,0.190455,0.108135,0.346574,0.071272,0.110362,0.039530
88,IDN,Indonesia,2025,Mt CO2eq/yr,1358.912941,140.711312,34.824594,344.914580,133.505650,384.327116,...,157.678552,78.298358,0.103547,0.025627,0.253817,0.098244,0.282820,0.062294,0.116033,0.057618
